# 09 — Export features and held-out logits to Zenodo (CPU, roughly 1–2 hours for ~10 GB; estimate)

Packages every frozen-feature file (`TMLR_v3/features/<dataset>/<encoder>.npy` + provenance `.json`) and every
per-pool held-out logits file (`results/pools/<dataset>_logits.npz`) into **one zip per dataset**. Then:

1. **Verifies** every archive by re-opening it and re-hashing every file inside it.
2. Writes `zenodo_manifest.json`, with the SHA-256 of every archive and of every file inside it, and a `README.md` for the record.
3. Optionally uploads everything to a **Zenodo draft**. It never publishes: you review the draft on zenodo.org and press *Publish* yourself, because publishing is irreversible.

Users get the data back with `python code/fetch_features.py --record <id> --root <dir>` from the GitHub repository. That
script re-checks every checksum.

**Licences: decide before uploading.** The features are derived from third-party datasets and models, several of them
gated (UNI, UNI2-h, Virchow2, H-optimus-0, Prov-GigaPath, Hibou-B), and some datasets have their own terms. Check each
model's and dataset's licence for whether redistributing derived features is allowed. Use `EXCLUDE_ENCODERS` /
`EXCLUDE_DATASETS` below, or `ACCESS_RIGHT = "restricted"`, for anything whose terms don't allow open redistribution.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, sys, importlib
PROJECT = "/content/drive/MyDrive/TMLR_v3"
CODE = os.path.join(PROJECT, "code")
os.makedirs(CODE, exist_ok=True)
if CODE not in sys.path:
    sys.path.insert(0, CODE)
print("project:", PROJECT)

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/export_zenodo.py
"""Package the frozen features and held-out logits for Zenodo, verify them, and
(optionally) upload them to a Zenodo *draft* that you review and publish yourself.

Layout produced (one archive per dataset keeps every file well under Zenodo's
limits and lets users download only what they need):

    features_<dataset>.zip   <dataset>/<encoder>.npy + <dataset>/<encoder>.json
    heldout_logits.zip       pools/<dataset>_logits.npz   (per-fold held-out logits)
    zenodo_manifest.json     SHA-256 of every archive and of every file inside it
    README.md                what the record contains and how to use it

The archives are written with ZIP_STORED: float32 arrays barely compress, and
stored members can be read back without decompression cost. Nothing is ever
modified in place; the source files on Drive are only read.
"""

from __future__ import annotations

import glob
import hashlib
import json
import os
import time
import zipfile

import config as CFG

CHUNK = 1 << 22


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for b in iter(lambda: fh.read(CHUNK), b""):
            h.update(b)
    return h.hexdigest()


def md5_file(path):
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for b in iter(lambda: fh.read(CHUNK), b""):
            h.update(b)
    return h.hexdigest()


# ---------------------------------------------------------------- inventory
def inventory(exclude_datasets=(), exclude_encoders=()):
    """Every feature file (with its provenance json) and every logits file."""
    feats = {}
    for npy in sorted(glob.glob(os.path.join(CFG.FEAT_DIR, "*", "*.npy"))):
        ds = os.path.basename(os.path.dirname(npy))
        enc = os.path.splitext(os.path.basename(npy))[0]
        if ds in exclude_datasets or enc in exclude_encoders or enc.startswith("_"):
            continue
        meta = npy[:-4] + ".json"
        feats.setdefault(ds, []).append({"encoder": enc, "npy": npy,
                                         "json": meta if os.path.exists(meta) else None,
                                         "bytes": os.path.getsize(npy)})
    logits = [p for p in sorted(glob.glob(os.path.join(CFG.RESULTS_DIR, "pools", "*_logits.npz")))
              if os.path.basename(p)[:-len("_logits.npz")] not in exclude_datasets]
    return feats, logits


def summary(feats, logits):
    n = sum(len(v) for v in feats.values())
    b = sum(x["bytes"] for v in feats.values() for x in v) + sum(os.path.getsize(p) for p in logits)
    return {"datasets": len(feats), "feature_files": n, "logits_files": len(logits),
            "total_gb": round(b / 1e9, 2),
            "per_dataset": {ds: {"encoders": len(v), "gb": round(sum(x["bytes"] for x in v) / 1e9, 3)}
                            for ds, v in sorted(feats.items())}}


# ---------------------------------------------------------------- build
def build(out_dir, feats, logits, include_logits=True, verbose=True):
    """Write one archive per dataset (+ logits) and the manifest. Resumable: an
    archive that already exists and verifies against its sidecar is skipped."""
    os.makedirs(out_dir, exist_ok=True)
    manifest = {"created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "archives": {}}
    jobs = [(f"features_{ds}.zip", [(x["npy"], f"{ds}/{x['encoder']}.npy") for x in items]
             + [(x["json"], f"{ds}/{x['encoder']}.json") for x in items if x["json"]])
            for ds, items in sorted(feats.items())]
    if include_logits and logits:
        jobs.append(("heldout_logits.zip", [(p, f"pools/{os.path.basename(p)}") for p in logits]))
    for name, members in jobs:
        path = os.path.join(out_dir, name)
        side = path + ".members.json"
        if os.path.exists(path) and os.path.exists(side):
            rec = json.load(open(side))
            if rec.get("archive_sha256") == sha256_file(path):
                manifest["archives"][name] = rec
                if verbose:
                    print(f"  {name}: exists, verified - skipped", flush=True)
                continue
        t0 = time.time()
        inner = {}
        with zipfile.ZipFile(path + ".part", "w", zipfile.ZIP_STORED, allowZip64=True) as z:
            for src, arc in members:
                inner[arc] = {"sha256": sha256_file(src), "bytes": os.path.getsize(src)}
                z.write(src, arc)
        os.replace(path + ".part", path)
        rec = {"archive_sha256": sha256_file(path), "archive_md5": md5_file(path),
               "archive_bytes": os.path.getsize(path), "members": inner}
        with open(side, "w") as fh:
            json.dump(rec, fh, indent=1)
        manifest["archives"][name] = rec
        if verbose:
            print(f"  {name}: {len(inner)} files, {rec['archive_bytes'] / 1e9:.2f} GB "
                  f"({(time.time() - t0) / 60:.1f} min)", flush=True)
    with open(os.path.join(out_dir, "zenodo_manifest.json"), "w") as fh:
        json.dump(manifest, fh, indent=1)
    return manifest


def verify(out_dir, verbose=True):
    """Re-open every archive and re-hash every member against the manifest."""
    man = json.load(open(os.path.join(out_dir, "zenodo_manifest.json")))
    bad = []
    for name, rec in man["archives"].items():
        path = os.path.join(out_dir, name)
        if sha256_file(path) != rec["archive_sha256"]:
            bad.append((name, "archive hash"))
            continue
        with zipfile.ZipFile(path) as z:
            names = set(z.namelist())
            for arc, info in rec["members"].items():
                if arc not in names:
                    bad.append((name, f"missing {arc}"))
                    continue
                h = hashlib.sha256()
                with z.open(arc) as fh:
                    for b in iter(lambda: fh.read(CHUNK), b""):
                        h.update(b)
                if h.hexdigest() != info["sha256"]:
                    bad.append((name, f"hash {arc}"))
        if verbose:
            print(f"  {name}: {'OK' if not any(b[0] == name for b in bad) else 'FAILED'}", flush=True)
    return bad


def write_readme(out_dir, title, github_url, paper_title, summary_dict):
    lines = [f"# {title}", "",
             f"Frozen-encoder features and per-fold held-out logits for the study *{paper_title}*.",
             "", f"Code, registration, per-pool records and the verification gate: {github_url}", "",
             "## Contents", "",
             f"- `features_<dataset>.zip`: one archive per dataset ({summary_dict['datasets']} datasets, "
             f"{summary_dict['feature_files']} feature files). Each holds `<dataset>/<encoder>.npy` (float32, "
             "rows in the order of the dataset manifest in the GitHub repository) and `<dataset>/<encoder>.json` "
             "(encoder source, preprocessing, pooling, load report, feature statistics, extraction time).",
             "- `heldout_logits.zip`: `pools/<dataset>_logits.npz`, the per-fold held-out probe logits of every "
             "encoder, with labels and fold masks, as used by the analysis.",
             "- `zenodo_manifest.json`: SHA-256 of every archive and of every file inside it.", "",
             "## Use", "",
             "`python code/fetch_features.py --record <this record's id> --root <dir>` in the GitHub repository "
             "downloads, verifies and unpacks everything into the layout the notebooks expect.", "",
             "## Provenance and licences", "",
             "Features were extracted from publicly available datasets with publicly released encoders, "
             "some of them gated, under their respective licences. Users must comply with the terms of the "
             "original datasets and models; see the GitHub repository for sources.", "",
             "## Per dataset", "", "| dataset | encoders | GB |", "|---|---|---|"]
    lines += [f"| {ds} | {v['encoders']} | {v['gb']} |" for ds, v in summary_dict["per_dataset"].items()]
    with open(os.path.join(out_dir, "README.md"), "w") as fh:
        fh.write("\n".join(lines) + "\n")


# ---------------------------------------------------------------- Zenodo
class Zenodo:
    """Minimal client for the Zenodo deposit REST API. Creates and fills a DRAFT;
    it never publishes (publishing on Zenodo is irreversible)."""

    def __init__(self, token, sandbox=False, session=None):
        import requests
        self.base = "https://sandbox.zenodo.org/api" if sandbox else "https://zenodo.org/api"
        self.web = self.base.replace("/api", "")
        self.s = session or requests.Session()
        self.s.headers["Authorization"] = f"Bearer {token}"

    def create_draft(self):
        r = self.s.post(f"{self.base}/deposit/depositions", json={})
        r.raise_for_status()
        return r.json()

    def get(self, dep_id):
        r = self.s.get(f"{self.base}/deposit/depositions/{dep_id}")
        r.raise_for_status()
        return r.json()

    def uploaded(self, dep):
        return {f["filename"]: f.get("checksum") for f in dep.get("files", [])}

    def upload(self, dep, path, verbose=True):
        """Stream one file into the draft's bucket; Zenodo returns its MD5, which is
        compared with the local file's."""
        name = os.path.basename(path)
        t0 = time.time()
        with open(path, "rb") as fh:
            r = self.s.put(f"{dep['links']['bucket']}/{name}", data=fh,
                           headers={"Content-Type": "application/octet-stream"})
        r.raise_for_status()
        remote = r.json().get("checksum", "")
        ok = remote.replace("md5:", "") == md5_file(path)
        if verbose:
            print(f"  uploaded {name} ({os.path.getsize(path) / 1e9:.2f} GB, "
                  f"{(time.time() - t0) / 60:.1f} min) checksum {'OK' if ok else 'MISMATCH'}", flush=True)
        if not ok:
            raise RuntimeError(f"Zenodo checksum mismatch for {name}")
        return remote

    def set_metadata(self, dep_id, metadata):
        r = self.s.put(f"{self.base}/deposit/depositions/{dep_id}", json={"metadata": metadata})
        if r.status_code >= 400:
            raise RuntimeError(f"metadata rejected ({r.status_code}): {r.text[:500]}")
        return r.json()


def upload_all(out_dir, token, metadata, sandbox=False, dep_id=None, verbose=True):
    """Upload every file in out_dir (except sidecars) to a draft. Re-running with
    the same dep_id skips files already uploaded with a matching checksum."""
    zc = Zenodo(token, sandbox=sandbox)
    dep = zc.get(dep_id) if dep_id else zc.create_draft()
    have = zc.uploaded(dep)
    files = sorted(p for p in glob.glob(os.path.join(out_dir, "*"))
                   if os.path.isfile(p) and not p.endswith((".members.json", ".part")))
    if len(files) > 100:
        raise RuntimeError(f"{len(files)} files: Zenodo allows at most 100 per record")
    for p in files:
        name = os.path.basename(p)
        if name in have and have[name] and have[name].replace("md5:", "") == md5_file(p):
            if verbose:
                print(f"  {name}: already in the draft - skipped", flush=True)
            continue
        zc.upload(dep, p, verbose=verbose)
    zc.set_metadata(dep["id"], metadata)
    url = f"{zc.web}/deposit/{dep['id']}"
    if verbose:
        print(f"\nDRAFT ready (not published): {url}")
    return {"id": dep["id"], "url": url}

In [ ]:
%%writefile /content/drive/MyDrive/TMLR_v3/code/fetch_features.py
"""Download the features and held-out logits from Zenodo, verify every checksum,
and set up a project folder in which notebooks 04-08 run unchanged.

    python code/fetch_features.py --record 1234567 --root ~/cka_study
    export V3_DRIVE=~/cka_study          # the notebooks' config reads this
    python code/verify_v3.py --root .    # still works from the repository alone

Resulting layout (the one the code expects under <root>/TMLR_v3):

    code/                       copied from this repository (hash-checked by the registration)
    prereg.json                 the registration
    manifests/                  row order, labels and groups of every dataset
    features/<dataset>/<encoder>.npy, .json
    results/pools/<dataset>.json, <dataset>_logits.npz
"""

from __future__ import annotations

import argparse
import glob
import hashlib
import json
import os
import shutil
import sys
import time
import zipfile

CHUNK = 1 << 22
REPO = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for b in iter(lambda: fh.read(CHUNK), b""):
            h.update(b)
    return h.hexdigest()


def record_files(record, sandbox=False, session=None):
    import requests
    s = session or requests.Session()
    base = "https://sandbox.zenodo.org" if sandbox else "https://zenodo.org"
    r = s.get(f"{base}/api/records/{record}", timeout=60)
    r.raise_for_status()
    out = []
    for f in r.json().get("files", []):
        name = f.get("key") or f.get("filename")
        link = (f.get("links") or {}).get("self") or (f.get("links") or {}).get("download")
        out.append({"name": name, "url": link, "size": f.get("size"), "checksum": f.get("checksum")})
    return out


def download(url, dst, session=None, retries=5):
    """Resumable download (HTTP Range); keeps <dst>.part until complete."""
    import requests
    s = session or requests.Session()
    part = dst + ".part"
    for attempt in range(retries):
        have = os.path.getsize(part) if os.path.exists(part) else 0
        try:
            with s.get(url, stream=True, timeout=120,
                       headers={"Range": f"bytes={have}-"} if have else {}) as r:
                if r.status_code == 416:
                    break
                r.raise_for_status()
                mode = "ab" if have and r.status_code == 206 else "wb"
                with open(part, mode) as fh:
                    for b in r.iter_content(CHUNK):
                        fh.write(b)
            break
        except Exception as e:
            print(f"  retry {attempt + 1}/{retries} for {os.path.basename(dst)}: {e}")
            time.sleep(10)
    os.replace(part, dst)
    return dst


def setup_project(root):
    """Copy what the notebooks need from this repository into <root>/TMLR_v3."""
    proj = os.path.join(root, "TMLR_v3")
    for d in ("code", "manifests", os.path.join("results", "pools"), "features", "logs"):
        os.makedirs(os.path.join(proj, d), exist_ok=True)
    for p in glob.glob(os.path.join(REPO, "code", "*.py")):
        shutil.copy2(p, os.path.join(proj, "code"))
    for p in glob.glob(os.path.join(REPO, "manifests", "*")):
        shutil.copy2(p, os.path.join(proj, "manifests"))
    for p in glob.glob(os.path.join(REPO, "results", "pools", "*.json")):
        shutil.copy2(p, os.path.join(proj, "results", "pools"))
    for p in ("battery.json", "paper_tables.json"):
        src = os.path.join(REPO, "results", p)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(proj, "results"))
    shutil.copy2(os.path.join(REPO, "registration", "prereg.json"), os.path.join(proj, "prereg.json"))
    return proj


def unpack(zip_path, proj, members_sha):
    """Extract an archive, re-hashing every member against the manifest."""
    with zipfile.ZipFile(zip_path) as z:
        for arc in z.namelist():
            if arc.startswith("pools/"):
                dst = os.path.join(proj, "results", arc)
            else:
                dst = os.path.join(proj, "features", arc)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            with z.open(arc) as src, open(dst + ".part", "wb") as out:
                shutil.copyfileobj(src, out, CHUNK)
            os.replace(dst + ".part", dst)
            want = members_sha.get(arc)
            if want and sha256_file(dst) != want:
                raise RuntimeError(f"checksum mismatch after unpacking {arc}")


def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--record", required=True, help="Zenodo record id")
    ap.add_argument("--root", required=True, help="folder to create the project in")
    ap.add_argument("--only", nargs="*", default=None,
                    help="dataset names to fetch (default: all); logits are always fetched")
    ap.add_argument("--sandbox", action="store_true")
    ap.add_argument("--keep-archives", action="store_true")
    a = ap.parse_args(argv)
    root = os.path.abspath(os.path.expanduser(a.root))
    dl = os.path.join(root, "zenodo_download")
    os.makedirs(dl, exist_ok=True)
    proj = setup_project(root)
    files = {f["name"]: f for f in record_files(a.record, a.sandbox)}
    if "zenodo_manifest.json" not in files:
        sys.exit("record has no zenodo_manifest.json")
    man_path = download(files["zenodo_manifest.json"]["url"], os.path.join(dl, "zenodo_manifest.json"))
    man = json.load(open(man_path))
    wanted = [n for n in man["archives"]
              if n == "heldout_logits.zip" or a.only is None
              or n[len("features_"):-len(".zip")] in set(a.only)]
    for name in wanted:
        dst = os.path.join(dl, name)
        rec = man["archives"][name]
        if not (os.path.exists(dst) and sha256_file(dst) == rec["archive_sha256"]):
            print(f"downloading {name} ({rec['archive_bytes'] / 1e9:.2f} GB) ...", flush=True)
            download(files[name]["url"], dst)
        if sha256_file(dst) != rec["archive_sha256"]:
            sys.exit(f"{name}: archive checksum mismatch")
        unpack(dst, proj, {k: v["sha256"] for k, v in rec["members"].items()})
        print(f"  {name}: verified and unpacked ({len(rec['members'])} files)", flush=True)
        if not a.keep_archives:
            os.remove(dst)
    print(f"\nproject ready: {proj}\nnext: export V3_DRIVE={root}  (then run notebooks 04-08 or code/ directly)")


if __name__ == "__main__":
    main()

## 1. Inventory and settings

In [ ]:
import config as CFG, export_zenodo as EZ
for m in (CFG, EZ):
    importlib.reload(m)

EXCLUDE_DATASETS = ()          # e.g. ("mhist",)        if a dataset's terms forbid redistribution
EXCLUDE_ENCODERS = ()          # e.g. ("virchow2", "uni") if a model's terms forbid redistributing its outputs
INCLUDE_LOGITS = True
OUT_DIR = "/content/zenodo_export"                          # local disk: fast; ~10 GB free needed
COPY_TO_DRIVE = False                                       # also keep a copy in TMLR_v3/zenodo_export/

feats, logits = EZ.inventory(EXCLUDE_DATASETS, EXCLUDE_ENCODERS)
S = EZ.summary(feats, logits)
print(f"{S['datasets']} datasets, {S['feature_files']} feature files, {S['logits_files']} logits files, {S['total_gb']} GB")
import shutil
print(f"free local disk: {shutil.disk_usage('/content').free / 1e9:.0f} GB")
import pandas as pd
pd.DataFrame(S["per_dataset"]).T

## 2. Build and verify the archives (resumable: finished archives are skipped)

In [ ]:
man = EZ.build(OUT_DIR, feats, logits, include_logits=INCLUDE_LOGITS)
problems = EZ.verify(OUT_DIR)
assert not problems, problems
print("\nall archives verified")

## 3. Record README (edit the GitHub URL first)

In [ ]:
GITHUB_URL = "https://github.com/<user>/<repository>"     # <- set before uploading
PAPER_TITLE = ("Evaluating Centered Kernel Alignment for Frozen-Encoder Selection in Logit-Level Fusion: "
               "A Pre-Specified Multi-Domain Study")
EZ.write_readme(OUT_DIR, "Frozen-encoder features and held-out logits", GITHUB_URL, PAPER_TITLE, S)
if COPY_TO_DRIVE:
    dst = os.path.join(PROJECT, "zenodo_export"); os.makedirs(dst, exist_ok=True)
    for f in os.listdir(OUT_DIR):
        shutil.copy2(os.path.join(OUT_DIR, f), dst)
print(sorted(os.listdir(OUT_DIR)))

## 4. Upload to a Zenodo draft (optional)
1. Create a token at zenodo.org → *Applications* → *Personal access tokens*, with scopes `deposit:write` and `deposit:actions`.
   **Test on the sandbox first**: sandbox.zenodo.org has separate accounts and separate tokens.
2. Add the token as a Colab secret named `ZENODO_TOKEN` (or `ZENODO_SANDBOX_TOKEN` for the sandbox).
3. Fill in the metadata below, choosing the licence deliberately, and run the cell. If the upload is interrupted, set
   `DEPOSITION_ID` to the id it printed and re-run: files already in the draft are skipped.

In [ ]:
from google.colab import userdata
SANDBOX = True                                             # test here first, then set False
DEPOSITION_ID = None                                       # set to resume an interrupted upload
token = userdata.get("ZENODO_SANDBOX_TOKEN" if SANDBOX else "ZENODO_TOKEN")

LICENSE = None            # REQUIRED, e.g. "cc-by-4.0" or "cc-by-nc-4.0" - check the source licences (see top)
ACCESS_RIGHT = "open"     # or "restricted" (then also set "access_conditions" below)
assert LICENSE, "choose a licence for the record before uploading"

metadata = {
    "title": "Frozen-encoder features and held-out logits for: " + PAPER_TITLE,
    "upload_type": "dataset",
    "description": open(os.path.join(OUT_DIR, "README.md")).read().replace("\n", "<br>"),
    "creators": [   # as on the paper; check names, affiliations and add ORCIDs ("orcid": "0000-...")
        {"name": "Munna, Mohammad Mehedi Hasan", "affiliation": "North South University"},
        {"name": "Kabir, Acramul Haque", "affiliation": "North South University"},
        {"name": "Tulona, Reefka Fabliha", "affiliation": "East West University"},
        {"name": "Antu, Nahid Hossain", "affiliation": "North South University"},
        {"name": "Ul Mulk, Md Mansur", "affiliation": "North South University"},
    ],
    "license": LICENSE,
    "access_right": ACCESS_RIGHT,
    "keywords": ["centered kernel alignment", "encoder selection", "model fusion", "representational similarity",
                 "computational pathology", "frozen features"],
    "related_identifiers": [{"identifier": GITHUB_URL, "relation": "isSupplementTo", "resource_type": "software"}],
}
res = EZ.upload_all(OUT_DIR, token, metadata, sandbox=SANDBOX, dep_id=DEPOSITION_ID)
print("Review the draft, then publish it on Zenodo. Record id for fetch_features.py:", res["id"])